#### Assignment

Nama : Beatrice Nathania
NIM : 2802490946

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark= SparkSession.builder.appName("student_perform").getOrCreate()

In [3]:
df = spark.read.format("csv").option("header", True).option("sep", ",").option("inferSchema", True).load("student_performance_updated_1000.csv")

In [4]:
df.show(10)

+---------+--------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+
|StudentID|    Name|Gender|AttendanceRate|StudyHoursPerWeek|PreviousGrade|ExtracurricularActivities|ParentalSupport|FinalGrade|Study Hours|Attendance (%)|Online Classes Taken|
+---------+--------+------+--------------+-----------------+-------------+-------------------------+---------------+----------+-----------+--------------+--------------------+
|      1.0|    John|  Male|          85.0|             15.0|         78.0|                      1.0|           High|      80.0|        4.8|          59.0|               false|
|      2.0|   Sarah|Female|          90.0|             20.0|         85.0|                      2.0|         Medium|      87.0|        2.2|          70.0|                true|
|      3.0|    Alex|  Male|          78.0|             10.0|         65.0|                      0.0|            Low|    

In [5]:
print("Jum Record : ", df.count())

Jum Record :  1000


In [6]:
df.columns

['StudentID',
 'Name',
 'Gender',
 'AttendanceRate',
 'StudyHoursPerWeek',
 'PreviousGrade',
 'ExtracurricularActivities',
 'ParentalSupport',
 'FinalGrade',
 'Study Hours',
 'Attendance (%)',
 'Online Classes Taken']

In [7]:
df.dtypes

[('StudentID', 'double'),
 ('Name', 'string'),
 ('Gender', 'string'),
 ('AttendanceRate', 'double'),
 ('StudyHoursPerWeek', 'double'),
 ('PreviousGrade', 'double'),
 ('ExtracurricularActivities', 'double'),
 ('ParentalSupport', 'string'),
 ('FinalGrade', 'double'),
 ('Study Hours', 'double'),
 ('Attendance (%)', 'double'),
 ('Online Classes Taken', 'boolean')]

In [8]:
df.printSchema()

root
 |-- StudentID: double (nullable = true)
 |-- Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- AttendanceRate: double (nullable = true)
 |-- StudyHoursPerWeek: double (nullable = true)
 |-- PreviousGrade: double (nullable = true)
 |-- ExtracurricularActivities: double (nullable = true)
 |-- ParentalSupport: string (nullable = true)
 |-- FinalGrade: double (nullable = true)
 |-- Study Hours: double (nullable = true)
 |-- Attendance (%): double (nullable = true)
 |-- Online Classes Taken: boolean (nullable = true)



In [9]:
col_drop = ["AttendanceRate", "ParentalSupport"]

df = df.drop(*col_drop)

In [10]:
col_select = ["StudentID", "Name", "Study Hours", "Online Classes Taken"]
df.select(*col_select).show(5, False)

+---------+-------+-----------+--------------------+
|StudentID|Name   |Study Hours|Online Classes Taken|
+---------+-------+-----------+--------------------+
|1.0      |John   |4.8        |false               |
|2.0      |Sarah  |2.2        |true                |
|3.0      |Alex   |4.6        |false               |
|4.0      |Michael|2.9        |false               |
|5.0      |Emma   |4.1        |true                |
+---------+-------+-----------+--------------------+
only showing top 5 rows



In [11]:
#casting kolom online classes jadi integer

df = df.withColumn("Online Classes Taken", col("Online Classes Taken").cast("int"))

In [12]:
#count missing value
df.select([
    count(when(col(c).isNull()|(col(c) == "")| isnan(c),c)).alias(c) for c in [x for x in df.columns if x != "Online Classes Taken"]
]).show()

+---------+----+------+-----------------+-------------+-------------------------+----------+-----------+--------------+
|StudentID|Name|Gender|StudyHoursPerWeek|PreviousGrade|ExtracurricularActivities|FinalGrade|Study Hours|Attendance (%)|
+---------+----+------+-----------------+-------------+-------------------------+----------+-----------+--------------+
|       40|  34|    48|               50|           33|                       43|        40|         24|            41|
+---------+----+------+-----------------+-------------+-------------------------+----------+-----------+--------------+



In [13]:
#one way frequencies

df.groupBy("Gender").count().sort(col("count").asc()).show()

+------+-----+
|Gender|count|
+------+-----+
|  NULL|   48|
|Female|  451|
|  Male|  501|
+------+-----+



In [14]:
df.groupBy("Gender").count().filter("`count` > 400").show()

+------+-----+
|Gender|count|
+------+-----+
|Female|  451|
|  Male|  501|
+------+-----+



In [15]:
df.groupBy("Gender").count().filter("`count` > 400").count()

2

In [16]:
#casting (Types before casting)
df.dtypes

[('StudentID', 'double'),
 ('Name', 'string'),
 ('Gender', 'string'),
 ('StudyHoursPerWeek', 'double'),
 ('PreviousGrade', 'double'),
 ('ExtracurricularActivities', 'double'),
 ('FinalGrade', 'double'),
 ('Study Hours', 'double'),
 ('Attendance (%)', 'double'),
 ('Online Classes Taken', 'int')]

In [17]:
#mutiple column

int = ["StudentID"]
float = ["StudyHoursPerWeek", "PreviousGrade", "FinalGrade", "Attendance (%)"]
Bool = ["Online Classes Taken"]

for i in int :
    df_change = df.withColumn(i, col(i).cast("int"))
for i in float :
    df_change = df_change.withColumn(i, col(i).cast("float"))
for i in Bool :
    df_change = df_change.withColumn(i, col(i).cast("boolean"))

In [18]:
# types after casting
df_change.dtypes

[('StudentID', 'int'),
 ('Name', 'string'),
 ('Gender', 'string'),
 ('StudyHoursPerWeek', 'float'),
 ('PreviousGrade', 'float'),
 ('ExtracurricularActivities', 'double'),
 ('FinalGrade', 'float'),
 ('Study Hours', 'double'),
 ('Attendance (%)', 'float'),
 ('Online Classes Taken', 'boolean')]

In [19]:
df_change.describe().show()

+-------+-----------------+--------------+------+------------------+------------------+-------------------------+-----------------+------------------+------------------+
|summary|        StudentID|          Name|Gender| StudyHoursPerWeek|     PreviousGrade|ExtracurricularActivities|       FinalGrade|       Study Hours|    Attendance (%)|
+-------+-----------------+--------------+------+------------------+------------------+-------------------------+-----------------+------------------+------------------+
|  count|              960|           966|   952|               950|               967|                      957|              960|               976|               959|
|   mean|5416.019791666667|          NULL|  NULL|17.630526315789474| 77.59875904860392|       1.5203761755485894|80.03020833333333|2.4069672131147564| 77.24817518248175|
| stddev|2653.748319352276|          NULL|  NULL| 6.272131709564998|10.006639540693408|       1.0464388027817804|9.493652121996377|  1.62026671528718|

In [20]:
#median calc
df_median = df.filter((col("FinalGrade") != 0 )&col("FinalGrade").isNotNull()& (~isnan("FinalGrade")))
df_median.count()

960

In [21]:
median = df_median.approxQuantile("FinalGrade", [0.5], 0.001)
print("The median of FinalGrade : ", str(median[0]))

The median of FinalGrade :  80.0


In [22]:
# Distinct Counts
df.agg(countDistinct(col("Name"))).alias("Name_Count").show()

+--------------------+
|count(DISTINCT Name)|
+--------------------+
|                 962|
+--------------------+



In [23]:
df.select("Name","Gender").distinct().show(5)

+---------------+------+
|           Name|Gender|
+---------------+------+
|Brittany Sutton|Female|
| Trevor Freeman|  Male|
|  Robert Taylor|  Male|
|    Belinda Kim|Female|
| Denise Coleman|  Male|
+---------------+------+
only showing top 5 rows



In [24]:
df.filter(df.Name.contains("ara")).show(5)

+---------+-------------+------+-----------------+-------------+-------------------------+----------+-----------+--------------+--------------------+
|StudentID|         Name|Gender|StudyHoursPerWeek|PreviousGrade|ExtracurricularActivities|FinalGrade|Study Hours|Attendance (%)|Online Classes Taken|
+---------+-------------+------+-----------------+-------------+-------------------------+----------+-----------+--------------+--------------------+
|      2.0|        Sarah|Female|             20.0|         85.0|                      2.0|      87.0|        2.2|          70.0|                   1|
|   8342.0|Sarah Johnson|  Male|             18.0|         90.0|                      0.0|      92.0|        1.5|          87.0|                   0|
|   2976.0|  Sarah Young|  Male|             20.0|         70.0|                      0.0|      92.0|        3.7|          67.0|                   0|
|   6931.0|   Sarah Ross|  Male|             12.0|         90.0|                      3.0|      72.0

In [37]:
mean_value = df.select(mean("FinalGrade").alias("Mean_FinalGrade")).collect()[0][0]
mean_value

80.03020833333333

In [40]:
df = df.withColumn("mean_FinalGrade", lit(mean_value))

df = df.withColumn("Variance",pow(col("FinalGrade")-col("mean_FinalGrade"),2))

In [50]:
# variance Calculation

varian_sum = df.agg({"Variance" : "sum"}).collect()[0][0]
print("Variance Sum = ", varian_sum)

variance_pop = df.select(var_pop("FinalGrade").alias("Variance_population")).collect()[0]["Variance_population"]
print(f"Variance Population = {variance_pop:.4f}")

Variance Sum =  86434.1239583334
Variance Population = 90.0355


In [26]:
import math

In [27]:
# standard deviation

std  = math.sqrt(variance_pop)
print(f"Standard Deviation = {std}")

Standard Deviation = 9.488706223186098


In [42]:
df.select("Name", "Study Hours", "FinalGrade", "mean_FinalGrade", "Variance").show(5)

+-------+-----------+----------+-----------------+-------------------+
|   Name|Study Hours|FinalGrade|  mean_FinalGrade|           Variance|
+-------+-----------+----------+-----------------+-------------------+
|   John|        4.8|      80.0|80.03020833333333|9.12543402777835E-4|
|  Sarah|        2.2|      87.0|80.03020833333333|   48.5779958767361|
|   Alex|        4.6|      68.0|80.03020833333333|  144.7259125434028|
|Michael|        2.9|      92.0|80.03020833333333| 143.27591254340277|
|   Emma|        4.1|      85.0|80.03020833333333| 24.698829210069434|
+-------+-----------+----------+-----------------+-------------------+
only showing top 5 rows

